In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = sns.load_dataset("penguins")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.isna().count()

In [ ]:
df_clean = df.drop_duplicates()

In [ ]:
categorical = df_clean.select_dtypes(include="object").columns
numerical = df_clean.select_dtypes(exclude="object").columns

In [ ]:
for col in numerical:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
for col in categorical:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [ ]:
df_clean.isnull().sum()

In [ ]:
biscoe_df = df[df.island == "Biscoe"]
df.shape

In [ ]:
heavy_males = df[(df.body_mass_g > df.body_mass_g.mean()) & (df.sex == "Male")]
heavy_males.head()

In [ ]:
df_clean = df_clean.sort_values(by=["body_mass_g", "bill_length_mm"], ascending=[False, True])
df_clean.head()

In [ ]:
df["bill_ratio"] = df.bill_length_mm / df.bill_depth_mm
df_clean["bill_ratio"] = df["bill_ratio"]

In [ ]:
df_clean["bill_ratio"].describe()

In [ ]:
sns.countplot(data=df_clean, x='species')

plt.title("Count of Penguins per Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.show()

In [ ]:
sns.scatterplot(data=df_clean, x="bill_length_mm", y="bill_depth_mm", hue="species")

In [ ]:
sns.boxplot(data=df_clean, x="body_mass_g", hue="species")

In [ ]:
numerical = df.select_dtypes(include='number')
sns.heatmap(data=numerical.corr(), annot=True, cmap="coolwarm")

In [ ]:
X = df_clean.drop("species", axis=1)
y = df_clean["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
knn_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

In [ ]:
knn_model.fit(X_train, y_train)

In [ ]:
predictions = knn_model.predict(X_test)
accuracy_score(y_test, predictions)

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

In [ ]:
logistic_model.fit(X_train, y_train)

In [ ]:
predictions_log = logistic_model.predict(X_test)
accuracy_score(y_test, predictions_log)

In [ ]:

numerical_data = df.select_dtypes(include="number")

sns.heatmap(data=numerical_data.corr(), annot=True, cmap="coolwarm")